## 3D flattop-source phase modulated (VarOptic) Gaussian focusing

* We generated a transmission function corresponding to thermal-profile related phase modulation.


* We create a seed beam propagation object called `mybeam` and give the initial parameters:

    - Initial mesh grid spacing \ layout.
    - The time evolution matrix.
    - The transient solution to the heat equation.
    - The normalised (to fd coefficients) initial heat load.

* Generate and Plot seed beam profile.

* Generate and Play animation (loop) for temporal seed beam evolution at fixed z.

<div class="alert alert-block alert-info"> **Info:** Due to the numerics of the convolution function, it will show signs of breakdown near the boundaries. To circumvent this use a bigger meshgrid.</div>

In [1]:
import os, sys
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

import importlib
import photonicslab

In [15]:
importlib.reload(photonicslab.heateqn)
mygauss3dQ = photonicslab.heateqn.HeatEqn3D(nx=100, ny=100, nz=50, nt=300, Lx=.7, Ly=.7, Lz=.5,\
                                            hdf5grp='nb11a', Qmax=100., gain_medium='tmyvo4', cavity_mode_waist=0.4)

mygauss3dQ.add_normalized_pump_gaussian(wp=0.15, alpha=2.90, flatness=2, q=60)
mygauss3dQ.add_normalized_pump_gaussian(wp=0.2, alpha=2.90, flatness=6, q=20)
# Check the above units
mygauss3dQ.plot_source_profile(savepath='images/nb11a_sourceq.png', z_ind=-1)
mygauss3dQ.get_cn_fd_params()

importlib.reload(photonicslab.fdsolver)
myfdsolver3d = photonicslab.fdsolver.FDSolver3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a')
myfdsolver3d.adi_method()

importlib.reload(photonicslab.visualize)
myview3d = photonicslab.visualize.Visualize3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a')
myview3d.steadystate_profiles(savepath='images/nb11a_steady_state.png')
myview3d.transient_profiles(savepath='videos/nb11a_transient.mp4')

importlib.reload(photonicslab.phasemod)
myphasemod3d = photonicslab.phasemod.ThermalLens3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a')
myphasemod3d.phase(hdf5table='temperature3d', hdf5dset_phase='phase3d',  hdf5dset_phase_collapsed='phase2d')
myphasemod3d.phase_heat(hdf5table='temperature3d', hdf5dset_phase='phase3d_heat',\
                        hdf5dset_phase_collapsed='phase2d_heat')

importlib.reload(photonicslab.visualize)
myphaseview3d = photonicslab.visualize.VisualizePhase3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a',\
                                                       hdf5table='phase3d')
myphaseview3d.steadystate_profiles(savepath='images/nb11a_steady_state_phase.png')
myphaseview3d.transient_profiles(savepath='videos/nb11a_transient_phase.mp4')

Pump side total input power: 9.47 W
Output coupler total power: 2.22 W
adi_method took 1837.0288479328156 seconds.
transient_profiles took 494.98474740982056 seconds.


/usr/lib/python3.7/site-packages/matplotlib/colorbar.py:600: UserWarning: set_ticks() must have been called.
  warnings.warn("set_ticks() must have been called.")


transient_profiles took 584.7915217876434 seconds.


<img width="800" height="600" src="images/nb11a_sourceq.png"/>
<img width="800" height="600" src="images/nb11a_steady_state.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb11a_transient.mp4"/>

<img width="800" height="600" src="images/nb11a_steady_state_phase.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb11a_transient_phase.mp4"/>

In [76]:
importlib.reload(photonicslab.visualize)
importlib.reload(photonicslab.beamprop)
mybeamviewA = photonicslab.visualize.VisualizeBeam2D(wavelen=1.064e-4, waist=0.275, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d')
mybeamviewA.steadystate_profiles(z_dist=250, savepath='images/nb11a_steady_state_beam_zA.png')
mybeamviewA.steadystate_profiles(z_dist=290, savepath='images/nb11a_steady_state_beam_zB.png')
mybeamviewA.steadystate_profiles(z_dist=330, savepath='images/nb11a_steady_state_beam_zC.png')
mybeamviewA.steadystate_profiles(z_dist=370, savepath='images/nb11a_steady_state_beam_zD.png')
mybeamviewA.steadystate_profiles(z_dist=410, savepath='images/nb11a_steady_state_beam_zE.png')
mybeamviewA.steadystate_profiles(z_dist=460, savepath='images/nb11a_steady_state_beam_zF.png')
mybeamviewA.steadystate_profiles(z_dist=500, savepath='images/nb11a_steady_state_beam_zG.png')
mybeamviewA.steadystate_profiles(z_dist=540, savepath='images/nb11a_steady_state_beam_zH.png')
mybeamviewA.transient_profiles(306, savepath='videos/nb11a_transient_beam_focus.mp4')

transient_profiles took 120.32160663604736 seconds.


In [77]:
importlib.reload(photonicslab.beamqual)
mymsquaredA = photonicslab.beamqual.MSquared(wavelen=1.064e-4, waist=0.275, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d', hdf5subgrp='w0275')
print(mymsquaredA.measure_focal_length(z_start=200, z_end=400))
print(mymsquaredA.measure_waist_first_null())
print(mymsquaredA.measure_waist_fwhm())
print(mymsquaredA.measure_waist_variance())
print(mymsquaredA.measure_quality(z_start=250, z_end=900, num_measurements=30, debug=True))
mymsquaredA.plot_quality_profile("images/nb11a_waist.png")

[306.351157091558, 306.351157091558]
[0.2121212121212121, 0.2121212121212121]
[0.05656565656565657, 0.04949494949494951]
[0.20722857573129458, 0.20720367244534613]
Wavelength: 0.0001064
Focal length in x: 306.351157091558
Focal length in y: 306.351157091558
Beam waist in x: 0.20722857573129458
Beam waist in y: 0.20720367244534613
Rayleigh range in x 5.07e+03
Rayleigh range in y 5.07e+03
Axial distance: 250.0
Variance_x: [0.33801646]
Variance_y: [0.33727217]
MSquared_x: [28.99577126]
MSquared_y: [28.89204109]
Axial distance: 272.41379310344826
Variance_x: [0.33750585]
Variance_y: [0.3366174]
MSquared_x: [48.02932888]
MSquared_y: [47.82398743]
Axial distance: 294.82758620689657
Variance_x: [0.34523906]
Variance_y: [0.34432627]
MSquared_x: [146.61563025]
MSquared_y: [146.00162211]
Axial distance: 317.2413793103448
Variance_x: [0.35744897]
Variance_y: [0.35657924]
MSquared_x: [163.63863914]
MSquared_y: [163.02893175]
Axial distance: 339.65517241379314
Variance_x: [0.36912212]
Variance_y: [

/home/nlisa/course/photonicslab/photonicslab/beamqual.py:398: UserWarning: AstigmaticBeamWarning: Non-symmetric beam waist.
  warnings.warn("AstigmaticBeamWarning: {0}".format(astigmatic_beam_warning), UserWarning)


Axial distance: 608.6206896551724
Variance_x: [0.41645794]
Variance_y: [0.41509266]
MSquared_x: [7.31236748]
MSquared_y: [7.27990438]
Axial distance: 631.0344827586207
Variance_x: [0.42235753]
Variance_y: [0.42103239]
MSquared_x: [6.93545407]
MSquared_y: [6.90621606]
Axial distance: 653.4482758620691
Variance_x: [0.42811267]
Variance_y: [0.42683022]
MSquared_x: [6.6037804]
MSquared_y: [6.57738608]
Axial distance: 675.8620689655172
Variance_x: [0.43363408]
Variance_y: [0.4323941]
MSquared_x: [6.30749031]
MSquared_y: [6.28357583]
Axial distance: 698.2758620689656
Variance_x: [0.43886967]
Variance_y: [0.43767002]
MSquared_x: [6.03965776]
MSquared_y: [6.0178886]
Axial distance: 720.6896551724138
Variance_x: [0.44379302]
Variance_y: [0.44263033]
MSquared_x: [5.79528818]
MSquared_y: [5.77536511]
Axial distance: 743.1034482758621
Variance_x: [0.44839438]
Variance_y: [0.44726459]
MSquared_x: [5.57067254]
MSquared_y: [5.55233325]
Axial distance: 765.5172413793103
Variance_x: [0.45267415]
Varian

/home/nlisa/course/photonicslab/photonicslab/beamqual.py:583: RuntimeWarning: invalid value encountered in sqrt
  
/home/nlisa/course/photonicslab/photonicslab/beamqual.py:584: RuntimeWarning: invalid value encountered in sqrt
  #grp[str('MSquared_x_fit')] = np.polyfit(self.z_range, grp[str('variance_x')], deg=2)


<img width="800" height="600" src="images/nb11a_steady_state_beam_zA.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zB.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zC.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zD.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zE.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zF.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zG.png"/>
<img width="800" height="600" src="images/nb11a_steady_state_beam_zH.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb11a_transient_beam_focus.mp4"/>

<img width="800" height="600" src="images/nb11a_waist.png"/>

In [74]:
importlib.reload(photonicslab.visualize)
importlib.reload(photonicslab.beamprop)
mybeamviewB = photonicslab.visualize.VisualizeBeam2D(wavelen=1.064e-4, waist=0.225, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d')
mybeamviewB.steadystate_profiles(z_dist=85, savepath='images/nb11a1_steady_state_beam_zA.png')
mybeamviewB.steadystate_profiles(z_dist=195, savepath='images/nb11a1_steady_state_beam_zB.png')
mybeamviewB.steadystate_profiles(z_dist=250, savepath='images/nb11a1_steady_state_beam_zC.png')
mybeamviewB.steadystate_profiles(z_dist=350, savepath='images/nb11a1_steady_state_beam_zD.png')
mybeamviewB.steadystate_profiles(z_dist=370, savepath='images/nb11a1_steady_state_beam_zE.png')
mybeamviewB.steadystate_profiles(z_dist=410, savepath='images/nb11a1_steady_state_beam_zF.png')
mybeamviewB.steadystate_profiles(z_dist=500, savepath='images/nb11a1_steady_state_beam_zG.png')
mybeamviewB.steadystate_profiles(z_dist=600, savepath='images/nb11a1_steady_state_beam_zH.png')
mybeamviewB.transient_profiles(298, savepath='videos/nb11a1_transient_beam_focus.mp4')

transient_profiles took 118.13587713241577 seconds.


In [75]:
importlib.reload(photonicslab.beamqual)
mymsquaredB = photonicslab.beamqual.MSquared(wavelen=1.064e-4, waist=0.225, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d', hdf5subgrp='w0225')
print(mymsquaredB.measure_focal_length(z_start=200, z_end=400))
print(mymsquaredB.measure_waist_first_null())
print(mymsquaredB.measure_waist_fwhm())
print(mymsquaredB.measure_waist_variance())
print(mymsquaredB.measure_quality(z_start=250, z_end=500, num_measurements=50, debug=True))
mymsquaredB.plot_quality_profile("images/nb11a1_waist.png")

[298.2811525464058, 298.2811525464058]
[0.20505050505050504, 0.20505050505050504]
[0.04949494949494951, 0.04949494949494951]
[0.17815121798642783, 0.17817131832780467]
Wavelength: 0.0001064
Focal length in x: 298.2811525464058
Focal length in y: 298.2811525464058
Beam waist in x: 0.17815121798642783
Beam waist in y: 0.17817131832780467
Rayleigh range in x 3.75e+03
Rayleigh range in y 3.75e+03
Axial distance: 250.0
Variance_x: [0.3069963]
Variance_y: [0.30621224]
MSquared_x: [27.23894255]
MSquared_y: [27.13548023]
Axial distance: 255.10204081632654
Variance_x: [0.30742615]
Variance_y: [0.30660081]
MSquared_x: [30.52177041]
MSquared_y: [30.39999686]
Axial distance: 260.2040816326531
Variance_x: [0.30815513]
Variance_y: [0.30729385]
MSquared_x: [34.73495062]
MSquared_y: [34.59095612]
Axial distance: 265.3061224489796
Variance_x: [0.30926031]
Variance_y: [0.30836982]
MSquared_x: [40.32516396]
MSquared_y: [40.1535201]
Axial distance: 270.40816326530614
Variance_x: [0.31076725]
Variance_y: [

/home/nlisa/course/photonicslab/photonicslab/beamqual.py:583: RuntimeWarning: invalid value encountered in sqrt
  
/home/nlisa/course/photonicslab/photonicslab/beamqual.py:584: RuntimeWarning: invalid value encountered in sqrt
  #grp[str('MSquared_x_fit')] = np.polyfit(self.z_range, grp[str('variance_x')], deg=2)


<img width="800" height="600" src="images/nb11a1_steady_state_beam_zA.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zB.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zC.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zD.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zE.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zF.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zG.png"/>
<img width="800" height="600" src="images/nb11a1_steady_state_beam_zH.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb11a1_transient_beam_focus.mp4"/>

<img width="800" height="600" src="images/nb11a1_waist.png"/>

In [72]:
importlib.reload(photonicslab.beamprop)
importlib.reload(photonicslab.visualize)

mybeamviewC = photonicslab.visualize.VisualizeBeam2D(wavelen=1.064e-4, waist=0.15, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d')
mybeamviewC.steadystate_profiles(z_dist=100, savepath='images/nb11a2_steady_state_beam_zA.png')
mybeamviewC.steadystate_profiles(z_dist=150, savepath='images/nb11a2_steady_state_beam_zB.png')
mybeamviewC.steadystate_profiles(z_dist=200, savepath='images/nb11a2_steady_state_beam_zC.png')
mybeamviewC.steadystate_profiles(z_dist=290, savepath='images/nb11a2_steady_state_beam_zD.png')
mybeamviewC.steadystate_profiles(z_dist=300, savepath='images/nb11a2_steady_state_beam_zE.png')
mybeamviewC.steadystate_profiles(z_dist=310, savepath='images/nb11a2_steady_state_beam_zF.png')
mybeamviewC.steadystate_profiles(z_dist=400, savepath='images/nb11a2_steady_state_beam_zG.png')
mybeamviewC.steadystate_profiles(z_dist=500, savepath='images/nb11a2_steady_state_beam_zH.png')
mybeamviewC.transient_profiles(300, savepath='videos/nb11a2_transient_beam_focus.mp4')

transient_profiles took 117.79164218902588 seconds.


In [78]:
importlib.reload(photonicslab.beamqual)
mymsquaredC = photonicslab.beamqual.MSquared(wavelen=1.064e-4, waist=0.15, p=0, l=0, z=0, phi=0, \
                                                    hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb11a', \
                                                    hdf5set_phase='phase2d', hdf5subgrp='w015')
print(mymsquaredC.measure_focal_length(z_start=200, z_end=320))
print(mymsquaredC.measure_waist_fwhm())
print(mymsquaredC.measure_waist_first_null())
print(mymsquaredC.measure_waist_variance())
print(mymsquaredC.measure_quality(z_start=100, z_end=900, num_measurements=150, debug=True))
mymsquaredC.plot_quality_profile("images/nb11a2_waist.png")

[206.43740713770967, 206.43740713770967]
[0.04949494949494951, 0.04949494949494951]
[0.21919191919191916, 0.21919191919191916]
[0.12541268481472542, 0.1256162544296499]
Wavelength: 0.0001064
Focal length in x: 206.43740713770967
Focal length in y: 206.43740713770967
Beam waist in x: 0.12541268481472542
Beam waist in y: 0.1256162544296499
Rayleigh range in x 1.86e+03
Rayleigh range in y 1.86e+03
Axial distance: 100.0
Variance_x: [0.18925129]
Variance_y: [0.1889404]
MSquared_x: [4.93083677]
MSquared_y: [4.91806094]
Axial distance: 116.3265306122449
Variance_x: [0.19464591]
Variance_y: [0.19426811]
MSquared_x: [6.11708525]
MSquared_y: [6.09957405]
Axial distance: 132.6530612244898
Variance_x: [0.20122458]
Variance_y: [0.20079867]
MSquared_x: [7.89745242]
MSquared_y: [7.87468129]
Axial distance: 148.9795918367347
Variance_x: [0.20832094]
Variance_y: [0.20785774]
MSquared_x: [10.72011581]
MSquared_y: [10.69009139]
Axial distance: 165.30612244897958
Variance_x: [0.21472131]
Variance_y: [0.21

<img width="800" height="600" src="images/nb11a2_steady_state_beam_zA.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zB.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zC.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zD.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zE.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zF.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zG.png"/>
<img width="800" height="600" src="images/nb11a2_steady_state_beam_zH.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb11a2_transient_beam_focus.mp4"/>

<img width="800" height="600" src="images/nb11a2_waist.png"/>